In [1]:
import json
from pathlib import Path
from imgnet.collections.store import IndexedDatasets
from imgnet.collections.source import HuggingFaceSource
from imgtools.nifti.crawl import Crawler
from rich import print
import pandas as pd

/home/gpudual/bhklab/declan/med-image_index/.pixi/envs/default/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Abdomen Atlas 3

COMPLETED!

I did have to manually extract 1 tar file using terminal but besides that it went smoothly.

In [2]:
DATASET_NAME = "Abdomen Atlas 3"

indexed_datasets = IndexedDatasets("indexed_datasets") # This will download the latest version of the indexed_datasets if you dont provide a path
print(indexed_datasets.collections[:10])



19:19:25 [info     ] Indexed datasets path: /home/gpudual/bhklab/declan/med-image_index/indexed_datasets [imgnet] call=store.__init__:65


[
    '4D-Lung',
    'ACRIN-6698',
    'ACRIN-Contralateral-Breast-MR',
    'ACRIN-FLT-Breast',
    'ACRIN-NSCLC-FDG-PET',
    'AMOS',
    'Abdomen Atlas 3',
    'Adrenal-ACC-Ki67-Seg',
    'Advanced-MRI-Breast-Lesions',
    'Anti-PD-1_Lung'
]

In [3]:
description = """
AbdomenAtlas 3.0 is the first public dataset with high quality abdominal CTs and paired radiology reports. The database includes more than 9,000 CT scans with radiology reports and per-voxel annotations of liver, kidney and pancreatic tumors.

Overall, AbdomenAtlas 3.0 presents per-voxel segmentation masks for:

    tumors in the kidney, pancreas, and liver
    blood vessels, including the aorta, IVC, SMA, CHA, CA, SA, portal vein, and splenic vein
    liver sub-segments
    pancreas sub-segments
    18 organs, including the gallbladder, left kidney, right kidney, spleen, stomach, left adrenal gland, right adrenal gland, bladder, colon, duodenum, esophagus, left femur, right femur, intestine, left lung, right lung, prostate, and rectum
"""

In [4]:
source = HuggingFaceSource(
    file_type='nifti',
    repo_id="AbdomenAtlas/AbdomenAtlas3.0Mini",
    description=description
)



In [5]:
print(source)

HuggingFaceSource(
    file_type=<FileType.NIFTI: 'nifti'>,
    post_download=['unzip'],
    source='huggingface',
    repo_id='AbdomenAtlas/AbdomenAtlas3.0Mini',
    description='\nAbdomenAtlas 3.0 is the first public dataset with high quality abdominal CTs and paired 
radiology reports. The database includes more than 9,000 CT scans with radiology reports and per-voxel annotations 
of liver, kidney and pancreatic tumors.\n\nOverall, AbdomenAtlas 3.0 presents per-voxel segmentation masks for:\n\n
tumors in the kidney, pancreas, and liver\n    blood vessels, including the aorta, IVC, SMA, CHA, CA, SA, portal 
vein, and splenic vein\n    liver sub-segments\n    pancreas sub-segments\n    18 organs, including the 
gallbladder, left kidney, right kidney, spleen, stomach, left adrenal gland, right adrenal gland, bladder, colon, 
duodenum, esophagus, left femur, right femur, intestine, left lung, right lung, prostate, and rectum\n'
)

In [6]:
dataset_index_path = (indexed_datasets.path / DATASET_NAME)
dataset_index_path.mkdir(parents=True, exist_ok=True)
(dataset_index_path / "parquet").mkdir(exist_ok=True)
with open(dataset_index_path / "source.json", "w") as f:
    json.dump(source.model_dump(mode="json"), f, indent=2)

In [7]:

print(indexed_datasets.source_config(DATASET_NAME))

HuggingFaceSource(
    file_type=<FileType.NIFTI: 'nifti'>,
    post_download=['unzip'],
    source='huggingface',
    repo_id='AbdomenAtlas/AbdomenAtlas3.0Mini',
    description='\nAbdomenAtlas 3.0 is the first public dataset with high quality abdominal CTs and paired 
radiology reports. The database includes more than 9,000 CT scans with radiology reports and per-voxel annotations 
of liver, kidney and pancreatic tumors.\n\nOverall, AbdomenAtlas 3.0 presents per-voxel segmentation masks for:\n\n
tumors in the kidney, pancreas, and liver\n    blood vessels, including the aorta, IVC, SMA, CHA, CA, SA, portal 
vein, and splenic vein\n    liver sub-segments\n    pancreas sub-segments\n    18 organs, including the 
gallbladder, left kidney, right kidney, spleen, stomach, left adrenal gland, right adrenal gland, bladder, colon, 
duodenum, esophagus, left femur, right femur, intestine, left lung, right lung, prostate, and rectum\n'
)

In [8]:
temp_data_path = Path("temp_data") / DATASET_NAME
temp_data_path.mkdir(parents=True, exist_ok=True)
#downloader = indexed_datasets.downloader(DATASET_NAME).download(temp_data_path)

In [9]:
import tarfile

def extract_all_tar_gz(folder_path, extract_to=None, remove_after=False):
    """
    Extract all .tar.gz archives in a folder.
    
    Args:
        folder_path (str): Path to folder containing .tar.gz files
        extract_to (str, optional): Directory to extract files to. 
                                    If None, extracts to same folder as each archive.
        remove_after (bool): If True, removes the archive after successful extraction.
    
    Returns:
        list: List of extracted file paths
    """
    folder = Path(folder_path)
    extracted_files = []
    
    # Get all .tar.gz files
    tar_gz_files = list(folder.glob("*.tar.gz"))
    
    if not tar_gz_files:
        print(f"No .tar.gz files found in {folder_path}")
        return extracted_files
    
    print(f"Found {len(tar_gz_files)} .tar.gz archives")
    
    for archive_path in tar_gz_files:
        try:
            # Determine extraction directory
            if extract_to:
                extract_dir = Path(extract_to)
            else:
                # Extract to same directory as archive, using archive name (without .tar.gz)
                extract_dir = folder / archive_path.stem.replace('.tar', '')
            
            # Create extraction directory if it doesn't exist
            extract_dir.mkdir(parents=True, exist_ok=True)
            
            print(f"Extracting: {archive_path.name} -> {extract_dir}")
            
            # Extract the archive
            with tarfile.open(archive_path, 'r:gz') as tar:
                tar.extractall(path=extract_dir)
                extracted_files.extend([str(extract_dir / member.name) for member in tar.getmembers()])
            
            # Remove archive if requested
            if remove_after:
                archive_path.unlink()
                print(f"Removed: {archive_path.name}")
                
        except Exception as e:
            print(f"Error extracting {archive_path.name}: {e}")
    
    return extracted_files

extract_all_tar_gz(temp_data_path / "image_only", remove_after=True)
extract_all_tar_gz(temp_data_path / "mask_only", remove_after=True)

No .tar.gz files found in temp_data/Abdomen Atlas 3/image_only

No .tar.gz files found in temp_data/Abdomen Atlas 3/mask_only

[]

In [10]:


# We have two different scan and mask patterns that i need to capture. We are going to do 2 crawls and combine the indexes.

crawler = Crawler(
    nifti_dir=temp_data_path,
    scan_name_pattern="image_only/AbdomenAtlas3_images_{drop}/BDMAP_{scan_number}/ct.nii.gz",
    mask_name_pattern="mask_only/AbdomenAtlas3_masks_{drop}/BDMAP_{scan_number}/segmentations/{ROINames}.nii.gz",
    output_dir=indexed_datasets.path,
    n_jobs=10,
    force=True,
    deep=False,
)
crawler.crawl()







In [11]:
# print(len(index_df))
# index_df

index_df = pd.read_csv(indexed_datasets.path / DATASET_NAME / "index.csv")

index_df.drop(columns=['drop'], inplace=True)

index_df

,scan_number,ROINames,filepath,file_type,reference_id,reference_scan
0,1,NaN,image_only/AbdomenAtlas3_images_BDMAP_BDMAP_00...,scan,BDMAP_BDMAP_00000001_BDMAP_00000232_00000001,NaN
1,2,NaN,image_only/AbdomenAtlas3_images_BDMAP_BDMAP_00...,scan,BDMAP_BDMAP_00000001_BDMAP_00000232_00000002,NaN
2,3,NaN,image_only/AbdomenAtlas3_images_BDMAP_BDMAP_00...,scan,BDMAP_BDMAP_00000001_BDMAP_00000232_00000003,NaN
3,4,NaN,image_only/AbdomenAtlas3_images_BDMAP_BDMAP_00...,scan,BDMAP_BDMAP_00000001_BDMAP_00000232_00000004,NaN
4,5,NaN,image_only/AbdomenAtlas3_images_BDMAP_BDMAP_00...,scan,BDMAP_BDMAP_00000001_BDMAP_00000232_00000005,NaN
...,...,...,...,...,...,...
407706,9262,rectum,mask_only/AbdomenAtlas3_masks_BDMAP_BDMAP_0000...,mask,BDMAP_BDMAP_00009049_BDMAP_00009262_00009262,image_only/AbdomenAtlas3_images_BDMAP_BDMAP_00...
407707,9262,spleen,mask_only/AbdomenAtlas3_masks_BDMAP_BDMAP_0000...,mask,BDMAP_BDMAP_00009049_BDMAP_00009262_00009262,image_only/AbdomenAtlas3_images_BDMAP_BDMAP_00...
407708,9262,stomach,mask_only/AbdomenAtlas3_masks_BDMAP_BDMAP_0000...,mask,BDMAP_BDMAP_00009049_BDMAP_00009262_00009262,image_only/AbdomenAtlas3_images_BDMAP_BDMAP_00...
407709,9262,superior_mesenteric_artery,mask_only/AbdomenAtlas3_masks_BDMAP_BDMAP_0000...,mask,BDMAP_BDMAP_00009049_BDMAP_00009262_00009262,image_only/AbdomenAtlas3_images_BDMAP_BDMAP_00...


In [12]:
def map_modality(row):
    if row["file_type"] == "scan":
            return "CT"
    else:
        return "SEG"
    
index_df["Modality"] = index_df.apply(map_modality, axis=1)
index_df["BodyPartExamined"] = "ABDOMEN"




In [13]:
meta_df = pd.read_csv(temp_data_path / "AbdomenAtlas3.0MiniWithMeta.csv")

In [14]:
meta_df['scan_number'] = meta_df['BDMAP ID'].str.extract(r'(\d+)').astype(pd.Int64Dtype)



/tmp/ipykernel_2594368/1659516969.py:1: UserWarning: Instantiating Int64Dtype without any arguments.Pass a Int64Dtype instance to silence this warning.
  meta_df['scan_number'] = meta_df['BDMAP ID'].str.extract(r'(\d+)').astype(pd.Int64Dtype)


In [16]:


print(index_df["scan_number"].dtype)

print(index_df.dtypes)

print(meta_df.dtypes)

full_df = index_df.join(meta_df, on="scan_number", how="left", lsuffix="", rsuffix="_meta")
full_df.drop(columns=["scan_number_meta"], inplace=True)

int64

scan_number          int64
ROINames            object
filepath            object
file_type           object
reference_id        object
reference_scan      object
Modality            object
BodyPartExamined    object
dtype: object

BDMAP ID                                                                                      object
spacing                                                                                       object
shape                                                                                         object
sex                                                                                           object
age                                                                                          float64
scanner                                                                                       object
contrast                                                                                      object
liver volume (cm^3)                                                                          float64
total liver lesion volume (cm^3)                                                             float64
number of liver lesion instances                                                             float64
largest liver lesion diameter (cm)                                                           float64
largest liver lesion location (1-8)                                                           object
largest liver lesion attenuation (hyperattenuating, isoattenuating, hypoattenuating)          object
pancreas volume (cm^3)                                                                       float64
total pancreatic lesion volume (cm^3)                                                        float64
number of pancreatic lesion instances                                                        float64
largest pancreatic lesion diameter (cm)                                                      float64
largest pancreatic lesion location (head, body, tail)                                         object
largest pancreatic lesion attenuation (hyperattenuating, isoattenuating, hypoattenuating)     object
left kidney volume (cm^3)                                                                    float64
right kidney volume (cm^3)                                                                   float64
kidney volume (cm^3)                                                                         float64
total kidney lesion volume (cm^3)                                                            float64
number of kidney lesion instances                                                            float64
largest kidney lesion diameter (cm)                                                          float64
largest kidney lesion location (left, right)                                                  object
largest kidney lesion attenuation (hyperattenuating, isoattenuating, hypoattenuating)         object
spleen volume (cm^3)                                                                         float64
total colon lesion volume (cm^3)                                                             float64
number of colon lesion instances                                                             float64
largest colon lesion diameter (cm)                                                           float64
largest colon lesion attenuation (hyperattenuating, isoattenuating, hypoattenuating)          object
radiologist note                                                                              object
structured report                                                                             object
narrative report                                                                              object
fusion structured report                                                                      object
fusion narrative report                                                                       object
scan_number                                                                                    Int64
dtype: object

In [17]:
from imgnet.collections.store import IndexedDatasets
from imgindex.model import validate_index
from imgindex.parquet import csv_to_parquet

index_df = full_df
index_df.rename(columns={"reference_id": "SampleID"}, inplace=True)
index_df["SampleID"] = index_df["SampleID"].astype(str)
index_df.to_csv(indexed_datasets.path / DATASET_NAME / "index.csv", index=False)
csv_to_parquet(index_df, indexed_datasets.path / DATASET_NAME)
validate_index(index_df, "nifti", lazy=True)

,scan_number,ROINames,filepath,file_type,SampleID,reference_scan,Modality,BodyPartExamined,BDMAP ID,spacing,...,spleen volume (cm^3),total colon lesion volume (cm^3),number of colon lesion instances,largest colon lesion diameter (cm),"largest colon lesion attenuation (hyperattenuating, isoattenuating, hypoattenuating)",radiologist note,structured report,narrative report,fusion structured report,fusion narrative report
0,1,NaN,image_only/AbdomenAtlas3_images_BDMAP_BDMAP_00...,scan,BDMAP_BDMAP_00000001_BDMAP_00000232_00000001,NaN,CT,ABDOMEN,BDMAP_00000002,[0.8984375 0.8984375 1. ],...,220.1,0.0,0.0,0.0,NaN,NaN,FINDINGS: \nSpleen: \nNormal size (volume: 220...,"The spleen is normal in size, measuring 220.1 ...",NaN,NaN
1,2,NaN,image_only/AbdomenAtlas3_images_BDMAP_BDMAP_00...,scan,BDMAP_BDMAP_00000001_BDMAP_00000232_00000002,NaN,CT,ABDOMEN,BDMAP_00000003,[1.5 1.5 1.5],...,372.4,0.0,0.0,0.0,NaN,Radiologist's notes: \nEnlargement of both lob...,CT Arterial Phase \n\nFINDINGS: \nSpleen: \nSp...,"The patient has an enlarged spleen, measuring ...",CT Arterial Phase \n\n\nFINDINGS: \nSpleen: \n...,"The patient has an enlarged spleen, measuring ..."
2,3,NaN,image_only/AbdomenAtlas3_images_BDMAP_BDMAP_00...,scan,BDMAP_BDMAP_00000001_BDMAP_00000232_00000003,NaN,CT,ABDOMEN,BDMAP_00000004,[0.703125 0.703125 5. ],...,180.9,16.9,1.0,3.4,hypoattenuating,NaN,CT Venous Phase \n\nFINDINGS: \nSpleen: \nNorm...,"The spleen, liver, pancreas, and kidneys are a...",NaN,NaN
3,4,NaN,image_only/AbdomenAtlas3_images_BDMAP_BDMAP_00...,scan,BDMAP_BDMAP_00000001_BDMAP_00000232_00000004,NaN,CT,ABDOMEN,BDMAP_00000005,[0.78125 0.78125 0.8 ],...,73.8,0.0,0.0,0.0,NaN,NaN,FINDINGS: \nSpleen: \nNormal size (volume: 73....,"The spleen is normal in size, measuring 73.8 c...",NaN,NaN
4,5,NaN,image_only/AbdomenAtlas3_images_BDMAP_BDMAP_00...,scan,BDMAP_BDMAP_00000001_BDMAP_00000232_00000005,NaN,CT,ABDOMEN,BDMAP_00000006,[0.78125 0.78125 2.5 ],...,113.7,0.0,0.0,0.0,NaN,NaN,CT Venous Phase \n\nFINDINGS: \nSpleen: \nNorm...,"The spleen is normal in size, measuring 113.7 ...",NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
407706,9262,rectum,mask_only/AbdomenAtlas3_masks_BDMAP_BDMAP_0000...,mask,BDMAP_BDMAP_00009049_BDMAP_00009262_00009262,image_only/AbdomenAtlas3_images_BDMAP_BDMAP_00...,SEG,ABDOMEN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
407707,9262,spleen,mask_only/AbdomenAtlas3_masks_BDMAP_BDMAP_0000...,mask,BDMAP_BDMAP_00009049_BDMAP_00009262_00009262,image_only/AbdomenAtlas3_images_BDMAP_BDMAP_00...,SEG,ABDOMEN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
407708,9262,stomach,mask_only/AbdomenAtlas3_masks_BDMAP_BDMAP_0000...,mask,BDMAP_BDMAP_00009049_BDMAP_00009262_00009262,image_only/AbdomenAtlas3_images_BDMAP_BDMAP_00...,SEG,ABDOMEN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
407709,9262,superior_mesenteric_artery,mask_only/AbdomenAtlas3_masks_BDMAP_BDMAP_0000...,mask,BDMAP_BDMAP_00009049_BDMAP_00009262_00009262,image_only/AbdomenAtlas3_images_BDMAP_BDMAP_00...,SEG,ABDOMEN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
